# Task 2B - 02 Rule engine and trip-time calculation
Before optimising anything we make the booklet's rules executable and test them against the booklet's own worked examples.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2b')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time
from task2b_common import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 120)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (METRICS, FIGURES, INTERIM): d.mkdir(parents=True, exist_ok=True)
scn, fleet, veh, dt, al = load_tables()
sub_team = pd.read_csv(SUBMISSION)            # the teammate's submitted allocation (read-only)
import task2b_exact as X
S = X.S


## 1. The feasibility rules (booklet p.20)
| # | Rule |
|---|---|
| R1 | all orders on one vehicle+trip share one brand and one district |
| R2 | chilled orders need a reefer (reefers may also carry ambient) |
| R3 | `van_only` outlets need a van |
| R4 | a vehicle serves only outlets of its own depot |
| R5 | whole orders: never split an order |
| R6 | per trip: total volume <= vehicle volume cap and total weight <= weight cap |
| R7 | at most 2 trips per vehicle; trip minutes fit the budgets: Fresh trips share **270 min**, Style+Tech trips share **480 min** |

`trip_minutes = depot_to_district_freeflow_min + inter_stop_freeflow_min x (orders - 1) + sum(service_allowance_min[brand, dock_type])` (no return leg).

## 2. Test the trip-time formula against the booklet's worked examples

In [2]:
allow = al.set_index(['brand', 'dock_type']).service_allowance_min; d = dt.set_index('district')
def minutes(district, brand, docks):
    return d.loc[district, 'depot_to_district_freeflow_min'] + d.loc[district, 'inter_stop_freeflow_min'] * (len(docks) - 1) + sum(allow[(brand, k)] for k in docks)
gampaha = minutes('Gampaha', 'Fresh', ['rear_dock', 'rear_dock', 'street'])
colombo = minutes('Colombo', 'Fresh', ['street'] * 4)
print('Gampaha trip (booklet example): %d min (booklet: 101)' % gampaha)
print('Colombo trip (booklet example): %d min (booklet: 112)' % colombo)
print('same vehicle, both Fresh trips: %d of 270 min (booklet: 213)' % (gampaha + colombo))
assert (gampaha, colombo) == (101, 112)
print('\nservice allowances (min per stop):'); print(allow.unstack().round(0))

Gampaha trip (booklet example): 101 min (booklet: 101)
Colombo trip (booklet example): 112 min (booklet: 112)
same vehicle, both Fresh trips: 213 of 270 min (booklet: 213)

service allowances (min per stop):
dock_type  mall_bay  rear_dock  street
brand                                 
Fresh            18         15      16
Style            59         38      46
Tech             55         43      55


## 3. The solver's rule engine agrees with an independent re-implementation

In [3]:
t = trip_table(sub_team, scn, veh, dt, al)          # independent: straight from the CSVs
mism = 0
for _, r in t.iterrows():
    orders = [S.ORDERS[x] for x in r.orders.split()]
    mism += abs(S.trip_minutes(orders) - r.trip_min) > 1e-9
print('trips checked:', len(t), '| minute mismatches between solver rule engine and independent calculation:', int(mism))
assert mism == 0

# unit tests of the rule predicates on synthetic situations
reefer_truck = S.VEHICLES_ALL['VEH003']; amb_truck = [v for v in S.AVAILABLE['S1'] if v.type == 'truck' and v.temp == 'ambient'][0]; van = [v for v in S.AVAILABLE['S1'] if v.type == 'van'][0]
chilled = next(o for o in S.ORDERS.values() if o.chilled and o.parking == 'normal'); vo = next(o for o in S.ORDERS.values() if o.parking == 'van_only' and not o.chilled)
tests = {'chilled on ambient truck is rejected': not S.compatible(amb_truck, chilled), 'chilled on reefer truck is accepted': S.compatible(reefer_truck, chilled),
         'van_only outlet on a truck is rejected': not S.compatible(amb_truck, vo), 'van_only outlet on a van is accepted': S.compatible(van, vo)}
pd.Series(tests)

trips checked: 23 | minute mismatches between solver rule engine and independent calculation: 0


chilled on ambient truck is rejected      True
chilled on reefer truck is accepted       True
van_only outlet on a truck is rejected    True
van_only outlet on a van is accepted      True
dtype: bool

## 4. Which orders can an available vehicle carry at all? (unavoidable deferrals)

In [4]:
avail_v = S.AVAILABLE['S1']
rows = []
for o in S.ORDERS.values():
    ok = [v.vid for v in avail_v if S.compatible(v, o) and o.m3 <= v.m3_cap + 1e-9 and o.kg <= v.kg_cap + 1e-9 and S.trip_minutes([o]) <= S.BUDGET[S.category(o.brand)]]
    rows.append((o.ref, o.outlet, o.brand, o.district, round(o.m3, 2), len(ok)))
c = pd.DataFrame(rows, columns=['order_ref', 'outlet', 'brand', 'district', 'm3', 'vehicles_that_can_carry_it_alone'])
print(c.vehicles_that_can_carry_it_alone.describe().round(1).to_dict())
print('\norders NO available vehicle can carry (unavoidable):'); c[c.vehicles_that_can_carry_it_alone == 0]

{'count': 85.0, 'mean': 18.9, 'std': 11.9, 'min': 0.0, '25%': 3.0, '50%': 28.0, '75%': 28.0, 'max': 28.0}

orders NO available vehicle can carry (unavoidable):


,order_ref,outlet,brand,district,m3,vehicles_that_can_carry_it_alone
78,S1-078,OUT070,Style,Kurunegala,40.66,0


**Finding.** Exactly one order is physically undeliverable today: S1-078 (OUT070, Style, 40.66 m3) is bigger than the largest compatible vehicle (38 m3). Orders cannot be split (R5), so this deferral is **unavoidable** regardless of policy. Everything else *can* be carried by at least one vehicle; whether it *is* depends on competition for scarce capacity.

## 5. How many stops fit in a Fresh trip? (district reach)

In [5]:
rows = []
for dist in d.index.intersection(scn.district.unique()):
    for dock in ('rear_dock', 'street'):
        a = allow[('Fresh', dock)]; out, inter = d.loc[dist, 'depot_to_district_freeflow_min'], d.loc[dist, 'inter_stop_freeflow_min']
        rows.append((dist, dock, int(out), int(inter), int(a), int(np.floor((270 - out + inter) / (inter + a)))))
r = pd.DataFrame(rows, columns=['district', 'dock', 'outbound_min', 'inter_stop_min', 'handling_min', 'max_stops_in_270_min'])
r.pivot(index='district', columns='dock', values='max_stops_in_270_min').join(d.depot_to_district_freeflow_min.rename('outbound_min')).sort_values('outbound_min')

,rear_dock,street,outbound_min
district,,,
Colombo,11,10,24
Gampaha,10,9,37
Kalutara,8,7,64
Galle,7,7,103
Kurunegala,4,4,127
Matara,5,5,137
Puttalam,3,3,173


**Finding.** A single Fresh trip to Puttalam leaves room for only a handful of stops and uses most of a vehicle's 270 minutes; nearby districts allow 9-10 stops. This is why *where* a chilled order is matters as much as how large it is.